# **Libs Install**

In [1]:
import sys
!{sys.executable} -m pip install ascon

# **Question - 5**

In [2]:
import sys    
import ascon

def ascon_sbox(x: int) -> int:
    if x < 0 or x >= 32:
        raise ValueError("x must be a 5-bit integer (0..31)")
    x0 = (x >> 0) & 1
    x1 = (x >> 1) & 1
    x2 = (x >> 2) & 1
    x3 = (x >> 3) & 1
    x4 = (x >> 4) & 1

    y0 = x4 ^ x0 ^ (x3 & x1)
    y1 = x0 ^ x1 ^ (x4 & x2)
    y2 = x1 ^ x2 ^ (x0 & x3)
    y3 = x2 ^ x3 ^ (x1 & x4)
    y4 = x3 ^ x4 ^ (x2 & x0)

    return (y0 << 0) | (y1 << 1) | (y2 << 2) | (y3 << 3) | (y4 << 4)

def TruthTable():
    return [ascon_sbox(i) for i in range(32)]

def Parity(v: int) -> int:
    return bin(v).count("1") & 1

def WalshMaxAbs(fvals):
    maxabs = 0
    argw = 0
    for w in range(32):
        s = 0
        for x in range(32):
            s += 1 if (fvals[x] ^ Parity(w & x)) == 0 else -1
        if abs(s) > maxabs:
            maxabs = abs(s)
            argw = w
    return maxabs, argw

def SboxNonlinearity(tt):
    min_nl = 10**9
    worst_a = 0
    worst_w = 0
    worst_maxabs = 0
    for a in range(1, 32):
        f = [Parity(a & tt[x]) for x in range(32)]
        maxabs, w = WalshMaxAbs(f)
        nl = 16 - (maxabs // 2)
        if nl < min_nl:
            min_nl = nl
            worst_a = a
            worst_w = w
            worst_maxabs = maxabs
    return min_nl, worst_a, worst_w, worst_maxabs

def IsInvolutory(tt):
    for x in range(32):
        y = tt[x]
        z = tt[y]
        if z != x:
            return False, x, y, z
    return True, None, None, None

def simplified_ascon_round(state: int, round_constant: int) -> int:
    if state < 0 or state >= 32:
        raise ValueError("state must be 5-bit (0..31)")
    if round_constant < 0 or round_constant >= 32:
        raise ValueError("round_constant must be 5-bit (0..31)")
    state = ascon_sbox(state) & 0x1F
    state = (state ^ ((state << 1) & 0x1F) ^ ((state >> 4) & 0x1F)) & 0x1F
    state = (state ^ round_constant) & 0x1F
    return state

def DdtCount(tt, dx: int, dy: int) -> int:
    dx &= 0x1F
    dy &= 0x1F
    c = 0
    for x in range(32):
        if (tt[x] ^ tt[x ^ dx]) == dy:
            c += 1
    return c

def AsconCiphertextFirst4Hex() -> str:

    key = b"0123456789ABCDEF"
    nonce = b"ABCDEFGHIJKLMNOP"
    plaintext = b"CRYPTOGRAPHYFUN!!"
    associateddata = None
    ad = b"" if associateddata is None else associateddata

    variant = "Ascon-128"

    if hasattr(ascon, "ascon_encrypt"):
        out = ascon.ascon_encrypt(key, nonce, ad, plaintext, variant=variant)
    elif hasattr(ascon, "encrypt"):
        out = ascon.encrypt(key, nonce, ad, plaintext, variant=variant)
    else:
        raise RuntimeError("ascon module missing encrypt function")

    if isinstance(out, tuple):
        ct = out[0]
        tag = out[1] if len(out) > 1 else b""
    else:
        if len(out) >= len(plaintext) + 16:
            ct = out[:-16]
            tag = out[-16:]
        else:
            ct = out
            tag = b""

    return ct[:4].hex()

tt = TruthTable()
print("=== Part 1(a): ascon_sbox(x) and Truth Table ===")
print("TruthTable =", tt)
print()

nl, worst_a, worst_w, worst_maxabs = SboxNonlinearity(tt)
invol_ok, ce_x, ce_sx, ce_ssx = IsInvolutory(tt)

print("=== Part 1(b): Properties ===")
print(f"Nonlinearity = {nl}")
print(f"WorstMaskA = {worst_a:05b} decimal {worst_a}")
print(f"WorstWalshW = {worst_w:05b} decimal {worst_w}")
print(f"MaxAbsWalsh = {worst_maxabs}")
if invol_ok:
    print("Involutory? = True")
else:
    print("Involutory? = False")
    print(f"Counterexample: x = {ce_x} S(x) = {ce_sx} S(S(x)) = {ce_ssx}")
print()

print("=== Part 2: simplified_ascon_round(state, round_constant) ===")
o1 = simplified_ascon_round(state=0b00101, round_constant=0b10000)
o2 = simplified_ascon_round(state=0b11111, round_constant=0b00001)
print(f"Input state=0b00101, rc=0b10000 -> output = {o1} binary {o1:05b}")
print(f"Input state=0b11111, rc=0b00001 -> output = {o2} binary {o2:05b}")
print()

print("=== Part 2: Official Ascon Library Encryption Check ===")
first4 = AsconCiphertextFirst4Hex()
print("CiphertextFirst4Bytes (8 hex chars):", first4)
print()

print("=== Part 4(a): DDT Entry for Δx=1 and Δy=1 ===")
cnt = DdtCount(tt, dx=0b00001, dy=0b00001)
print("Count =", cnt)

=== Part 1(a): ascon_sbox(x) and Truth Table ===
TruthTable = [0, 3, 6, 5, 12, 31, 10, 25, 24, 31, 31, 24, 20, 3, 19, 4, 17, 18, 31, 28, 31, 12, 17, 2, 9, 14, 6, 1, 7, 16, 8, 31]

=== Part 1(b): Properties ===
Nonlinearity = 8
WorstMaskA = 00001 decimal 1
WorstWalshW = 10001 decimal 17
MaxAbsWalsh = 16
Involutory? = False
Counterexample: x = 1 S(x) = 3 S(S(x)) = 5

=== Part 2: simplified_ascon_round(state, round_constant) ===
Input state=0b00101, rc=0b10000 -> output = 16 binary 10000
Input state=0b11111, rc=0b00001 -> output = 1 binary 00001

=== Part 2: Official Ascon Library Encryption Check ===
CiphertextFirst4Bytes (8 hex chars): 7037a3a2

=== Part 4(a): DDT Entry for Δx=1 and Δy=1 ===
Count = 0
